# **Кластеризация данных (Clustering Analysis)**

* __Цель кластеризации:__ Разбить совокупность пациентов на однородные группы (кластеры) на основе их медицинских показателей.
* __Задачи кластеризации:__ Применить алгоритмы обучения без учителя (Unsupervised Learning) для поиска скрытых паттернов в данных о пациентах.
* __Алгоритм использования:__
  1. __Метод K-средних (K-Means)__
  2. __Алгоритм DBSCAN (Пространственная кластеризация при наличии шума)__
  3. __Иерархическая кластеризация (Агломеративный подход)__

---

## **1. Подготовка данных к кластеризации**
* __Цель:__ Сформировать корректную, очищенную матрицу признаков для объективного обучения алгоритмов кластеризации "без учителя".
* __Задачи:__
  * Исключить из выборки целевую переменную, чтобы алгоритмы не имели к ней доступа при поиске скрытых паттернов.
  * Проверить итоговую матрицу на целостность (отсутствие пропущенных значений).
  * Оценить описательную статистику главных компонент (PCA) для подтверждения корректности масштабирования.

In [ ]:
# --- ИМПОРТ БИБЛИОТЕК И ПЕРВИЧНАЯ НАСТРОЙКА ---

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Установка параметра для воспроизводимости результатов
RANDOM_STATE = 42

# Настройка стиля графиков
sns.set_theme(
    style="whitegrid",
    palette="muted",
    rc={"figure.figsize": (10, 6), "axes.titlesize": 14},
)

In [ ]:
# --- 1. ЗАГРУЗКА ДАННЫХ ---

PROCESSED_DIR = Path("../data/processed")
INTERIM_DIR = Path("../data/interim")

data_pca = pd.read_csv(PROCESSED_DIR / "heart_disease_uci_pca.csv")
data_original = pd.read_csv(INTERIM_DIR / "heart_disease_uci_cleaned.csv")

In [ ]:
# --- 2. ПОДГОТОВКА МАТРИЦЫ ПРИЗНАКОВ ---

display(Markdown("#### **Формирование матрицы X**"))

# Отделение признаков (X) от целевой переменной / меток (y)
X_cluster = data_pca.drop(columns=["num"]).copy()
y_true = data_pca["num"].copy()

# Проверка отсутствия пропусков и корректности данных
missing_values = X_cluster.isna().sum().sum()

display(
    Markdown(
        f"* Матрица признаков `X` успешно сформирована.\n"
        f"* Размерность матрицы признаков `X`: **{X_cluster.shape[0]} строк и {X_cluster.shape[1]} главных компонент (PC)**.\n"
        f"* Пропущенных значений в данных PCA: **{missing_values}**."
    )
)

# Проверка распределения (PCA-компоненты должны иметь среднее ~0)
display(Markdown("#### **Описательная статистика признаков (X_cluster)**"))
display(X_cluster.describe().round(4))

## **2. Кластеризация методом K-Means**

### **2.1. Определение оптимального числа кластеров**

* __Цель:__ Определить математически обоснованное и практически интерпретируемое количество кластеров ($K$) для обучения модели K-Means.
* __Задачи:__
  1. Рассчитать внутрикластерную дисперсию (инерцию) для различных значений $K$ (Метод локтя).
  2. Вычислить коэффициент силуэта для оценки плотности и обособленности кластеров при различных $K$.
  3. Визуализировать метрики и выбрать оптимальное $K$ на основе консенсуса алгоритмов.

Главная проблема K-Means — выбор *K*. Методы для определения *K*:
* **Метод локтя (Elbow Method).** Оценивает сумму квадратов расстояний от точек до центроидов своих кластеров (инерцию). Оптимальным считается значение $K$, при котором на графике образуется характерный изгиб ("локоть"), где скорость уменьшения ошибки резко замедляется.
* **Коэффициент силуэта (Silhouette Score).** Измеряет плотность кластеров и расстояние между ними (принимает значения от -1 до 1). Чем ближе значение к 1, тем лучше объекты сгруппированы. Оптимальным считается $K$, при котором коэффициент достигает максимума.

In [ ]:
# --- 1. РАСЧЕТ МЕТРИК ---

# Списки для сохранения метрик
inertia_values = []
silhouette_scores = []

# Диапазон значений K
K_range = range(1, 11)
K_range_sil = range(2, 11)

display(
    Markdown("#### *Вычисление метрик (Инерция и Силуэт) для различных значений K...*")
)

for k in K_range:
    # Инициализация и обучение модели
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(X_cluster)

    # Сохранение инерции (Метод локтя)
    inertia_values.append(kmeans.inertia_)

    # Расчет коэффициента силуэта (K > 1)
    if k > 1:
        silhouette_avg = silhouette_score(X_cluster, kmeans.labels_)
        silhouette_scores.append(silhouette_avg)

# --- 2. ВИЗУАЛИЗАЦИЯ ---

plt.figure(figsize=(18, 6))

# График 1: Метод локтя (Elbow Method)
plt.subplot(1, 2, 1)
sns.lineplot(
    x=list(K_range),
    y=inertia_values,
    marker="o",
    color="teal",
    linewidth=2,
    markersize=8,
)
plt.title("Метод локтя (Elbow Method)", fontsize=14, pad=10)
plt.xlabel("Количество кластеров (K)", fontsize=12)
plt.ylabel("Инерция (Inertia)", fontsize=12)
plt.xticks(list(K_range))
plt.grid(True, linestyle="--", alpha=0.6)

# График 2: Коэффициент силуэта (Silhouette Score)
plt.subplot(1, 2, 2)
sns.lineplot(
    x=list(K_range_sil),
    y=silhouette_scores,
    marker="s",
    color="coral",
    linewidth=2,
    markersize=8,
)
plt.title("Коэффициент силуэта (Silhouette Score)", fontsize=14, pad=10)
plt.xlabel("Количество кластеров (K)", fontsize=12)
plt.ylabel("Средний коэффициент силуэта", fontsize=12)
plt.xticks(list(K_range_sil))
plt.grid(True, linestyle="--", alpha=0.6)

# Автоматическое нахождение и выделение максимума на графике силуэта
max_sil_index = np.argmax(silhouette_scores)
optimal_k_sil = K_range_sil[max_sil_index]
max_sil_value = silhouette_scores[max_sil_index]

plt.plot(optimal_k_sil, max_sil_value, marker="*", markersize=18, color="crimson")
plt.annotate(
    f" Максимум\n (K={optimal_k_sil})",
    xy=(optimal_k_sil, max_sil_value),
    xytext=(optimal_k_sil + 0.3, max_sil_value - 0.01),
    fontsize=12,
    color="darkred",
    weight="bold",
)

plt.tight_layout()
plt.show()

#### **Вывод по определению оптимального числа кластеров**

* **Анализ инерции (Метод локтя):** На левом графике наиболее выраженный излом ("локоть") наблюдается в диапазоне $K=2$ и $K=3$. Начиная с $K=3$, скорость снижения внутрикластерной дисперсии существенно падает, что говорит о нецелесообразности дальнейшего дробления данных.
* **Анализ плотности (Коэффициент силуэта):** Правый график демонстрирует однозначный глобальный максимум при $K=2$ (коэффициент силуэта $\approx 0.177$). Это математически подтверждает, что разбиение на две группы обеспечивает наилучшую сепарацию объектов в многомерном пространстве.
* **Итоговое обоснование:** Основываясь на консенсусе обеих метрик, для обучения алгоритма K-Means зафиксировано значение **$K=2$**. Данное математическое решение имеет сильный медицинский смысл в контексте данного набора данных: алгоритм обучения без учителя самостоятельно обнаружил скрытое бинарное разделение пациентов на две ключевые когорты — условную "группу нормы" и "группу риска".

### **2.2. Применение алгоритма K-Means**

* **Цель:** Выполнить сегментацию пациентов на основе выбранного оптимального значения $K$.
* **Задачи:**
  1. Инициализировать и обучить модель K-Means с заданными гиперпараметрами.
  2. Получить метки кластеров для каждого пациента.
  3. Визуализировать полученные кластеры в 2D-пространстве главных компонент (PCA) для наглядной оценки разделения.

In [ ]:
# --- ОБУЧЕНИЕ МОДЕЛИ ---

display(Markdown("#### **Обучение модели K-Means**"))

OPTIMAL_K = optimal_k_sil

# Инициализация и обучение модели
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
kmeans_final.fit(X_cluster)

# Получение меток кластеров
cluster_labels = kmeans_final.labels_

# display(Markdown(f"* Модель успешно обучена. Получены метки для **{len(cluster_labels)}** объектов."))

display(
    Markdown(
        f"* Модель успешно обучена на **{OPTIMAL_K}** кластерах.\n"
        f"* Получены метки для **{len(cluster_labels)}** объектов."
    )
)

### **2.3. Визуализация и анализ**

* **Цель:** Наглядно оценить качество разделения и сформировать детальные профили полученных групп пациентов.
* **Задачи:**
  1. Визуализировать кластеры в 2D-проекции PCA с акцентом на центроиды.
  2. Рассчитать количественные характеристики сегментов (размер, доля).
  3. Проанализировать средние значения клинических показателей на исходных данных.
  4. Сформулировать выводы о медицинских профилях групп.

In [ ]:
# --- 1. ВИЗУАЛИЗАЦИЯ КЛАСТЕРОВ ---

display(Markdown("#### **Визуализация кластеров K-Means в 2D-пространстве**"))

# Создание копии X_cluster для графика и добавление меток
plot_df = X_cluster.copy()
plot_df["Cluster"] = cluster_labels.astype(str)

plt.figure(figsize=(12, 6))
sns.scatterplot(
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="Set1",
    data=plot_df,
    alpha=0.7,
    edgecolor="k",
)

# Добавление центроидов на график
centroids_pca = kmeans_final.cluster_centers_
plt.scatter(
    centroids_pca[:, 0],
    centroids_pca[:, 1],
    marker="X",
    s=100,
    c="black",
    label="Центроиды",
    edgecolor="white",
    linewidth=2,
)

plt.title("2D-проекция кластеров K-Means (PC1 vs PC2)", fontsize=14)
plt.xlabel("Главная компонента 1 (PC1)", fontsize=12)
plt.ylabel("Главная компонента 2 (PC2)", fontsize=12)
plt.legend(title="Кластер (K-Means)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. ИНТЕРПРЕТАЦИЯ КЛАСТЕРОВ ---

display(Markdown("#### **Интерпретация кластеров**"))

# Создание копии оригинальных данных
data_profile = data_original.copy()

# Добавление полученных меток кластеров
data_profile["Cluster_KMeans"] = cluster_labels

# Расчет размеров кластеров
cluster_sizes = data_profile["Cluster_KMeans"].value_counts().sort_index()
cluster_percentages = (cluster_sizes / len(data_profile) * 100).round(1)

size_df = pd.DataFrame(
    {"Количество пациентов": cluster_sizes, "Доля (%)": cluster_percentages}
).reset_index()

display(Markdown("* **Количественное распределение:**"))
display(size_df)

# Исключение неинформативных столбцов перед расчетом средних значений признаков
cols_to_drop = ["id", "cluster"]
existing_drops = [col for col in cols_to_drop if col in data_profile.columns]

# Расчет средних значений признаков для каждого кластера
cluster_means = (
    data_profile.drop(columns=existing_drops)
    .groupby("Cluster_KMeans")
    .mean(numeric_only=True)
    .round(2)
)

display(Markdown("* **Средние значения медицинских показателей по кластерам:**"))
display(cluster_means.T)

#### **Выводы и описание сегментов (K-Means)**

**1. Визуализация (Разделимость кластеров):**
* На 2D-проекции отчетливо видно, что алгоритм K-Means успешно разделил пациентов на два крупных облака.
* Основная дисперсия и разделение происходят вдоль оси первой главной компоненты (PC1). 
* Несмотря на естественное для медицинских данных пересечение (overlap) в пограничной зоне, центроиды кластеров (черные кресты) находятся на значительном расстоянии друг от друга, что подтверждает математически качественную группировку.

**2. Интерпретация (Медицинские профили пациентов):**
Анализ средних значений исходных признаков позволяет составить четкие клинические портреты полученных групп:

* **Кластер 0 (Группа высокого риска / Пациенты с патологиями):**
  * **Размер:** 516 пациентов (56.2%).
  * **Профиль:** Пациенты старшей возрастной группы (средний возраст ~59 лет).
  * **Симптоматика:** Наблюдается повышенное артериальное давление в покое (`trestbps` = 134.5). Максимальная частота сердечных сокращений при нагрузке заметно снижена (`thalch` = 126). 
  * **Критические маркеры:** Более половины пациентов (54%) испытывают стенокардию, вызванную физической нагрузкой (`exang`), а показатель депрессии сегмента ST (`oldpeak`) критически завышен (1.22).
  * **Валидация (Диагноз):** Фактический средний показатель тяжести болезни (`num` = 1.45). *Хотя этот признак был скрыт от алгоритма*, K-Means абсолютно верно собрал в данный кластер людей с подтвержденными сердечно-сосудистыми заболеваниями.

* **Кластер 1 (Группа низкого риска / Условно здоровые):**
  * **Размер:** 402 пациента (43.8%).
  * **Профиль:** Более молодые пациенты (средний возраст ~47 лет).
  * **Симптоматика:** Давление находится в пределах нормы (`trestbps` = 125.7). Сердечно-сосудистая система отлично справляется с нагрузками: средний максимальный пульс высокий (`thalch` = 153).
  * **Критические маркеры:** Лишь 14% пациентов отмечают стенокардию при нагрузке, показатель `oldpeak` минимален (0.25).
  * **Валидация (Диагноз):** Показатель `num` (0.41) подтверждает, что модель "вслепую" выделила группу преимущественно здоровых пациентов.